# Baby Dragon Hatchling (BDH) - Google Colab Tutorial

Welcome! This notebook will help you:
- Train a biologically-inspired language model
- Understand how BDH works
- Experiment with the architecture
- Generate text with your trained model

**Hardware:** This notebook uses GPU acceleration (free on Colab)

**Time:** ~15-30 minutes for complete training

## Step 1: Setup Environment

First, let's check what GPU we have and clone the BDH repository.

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Clone the BDH repository
!git clone https://github.com/pathwaycom/bdh.git
%cd bdh

In [ ]:
# Install dependencies (torch, numpy, requests)
!pip install -r requirements.txt -q

## Step 2: Understand the Architecture

Let's explore the BDH model architecture before training.

In [ ]:
import bdh
import torch

# Create a BDH model with default configuration
config = bdh.BDHConfig()
model = bdh.BDH(config)

# Print model architecture
print("=" * 60)
print("BDH Model Architecture")
print("=" * 60)
print(f"Vocabulary size: {config.vocab_size}")
print(f"Hidden dimension: {config.n_embd}")
print(f"Number of layers: {config.n_layer}")
print(f"Dropout: {config.dropout}")
print("=" * 60)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / 1024 / 1024:.2f} MB (float32)")

## Step 3: Quick Training Run (Default)

Let's run the default training script to see BDH in action.

In [ ]:
# Run default training (3000 iterations)
!python train.py

## Step 4: Custom Training with Visualization

Now let's train with more control and visualize the learning process.

In [ ]:
import os
import numpy as np
import requests
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from contextlib import nullcontext

# Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BLOCK_SIZE = 512
BATCH_SIZE = 32
MAX_ITERS = 5000  # More iterations for better results
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.1
LOG_FREQ = 100

print(f"Training on: {device}")

# Download dataset if needed
input_file_path = "input.txt"
if not os.path.exists(input_file_path):
    data_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    with open(input_file_path, "w") as f:
        f.write(requests.get(data_url).text)
    print("Dataset downloaded!")

# Data loading function
def get_batch(split):
    data = np.memmap(input_file_path, dtype=np.uint8, mode="r")
    if split == "train":
        data = data[: int(0.9 * len(data))]
    else:
        data = data[int(0.9 * len(data)) :]
    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([torch.from_numpy((data[i : i + BLOCK_SIZE]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i + 1 : i + 1 + BLOCK_SIZE]).astype(np.int64)) for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

# Initialize model
config = bdh.BDHConfig()
model = bdh.BDH(config).to(device)
model = torch.compile(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Training loop with tracking
losses = []
steps = []

print("\nStarting training...")
print("=" * 60)

for step in range(MAX_ITERS):
    x, y = get_batch("train")
    
    logits, loss = model(x, y)
    
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    
    if step % LOG_FREQ == 0:
        losses.append(loss.item())
        steps.append(step)
        print(f"Step: {step}/{MAX_ITERS} | Loss: {loss.item():.4f}")

print("=" * 60)
print("Training complete!")

## Step 5: Visualize Training Progress

In [ ]:
# Plot loss curve
plt.figure(figsize=(10, 5))
plt.plot(steps, losses, linewidth=2)
plt.xlabel('Training Step', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('BDH Training Loss', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nInitial Loss: {losses[0]:.4f}")
print(f"Final Loss: {losses[-1]:.4f}")
print(f"Loss Reduction: {(1 - losses[-1]/losses[0]) * 100:.1f}%")

## Step 6: Text Generation

Now let's generate text with different prompts!

In [ ]:
model.eval()

def generate_text(prompt, max_tokens=200, temperature=0.8, top_k=10):
    """Generate text from a prompt."""
    prompt_tensor = torch.tensor(
        bytearray(prompt, "utf-8"), 
        dtype=torch.long, 
        device=device
    ).unsqueeze(0)
    
    with torch.no_grad():
        output = model.generate(
            prompt_tensor, 
            max_new_tokens=max_tokens, 
            top_k=top_k
        )
    
    result = bytes(output.to(torch.uint8).cpu().squeeze(0)).decode(errors='backslashreplace')
    return result

# Test different prompts
prompts = [
    "To be or not to be",
    "What is love",
    "The king said",
    "Once upon a time"
]

print("=" * 60)
print("TEXT GENERATION EXAMPLES")
print("=" * 60)

for prompt in prompts:
    print(f"\nPrompt: '{prompt}'")
    print("-" * 60)
    result = generate_text(prompt, max_tokens=150, top_k=5)
    print(result)
    print("=" * 60)

## Step 7: Interactive Text Generation

In [ ]:
# Interactive generation - change the prompt and run this cell
YOUR_PROMPT = "To be or "  # <-- Edit this!
MAX_TOKENS = 200           # <-- Edit this!
TOP_K = 5                  # <-- Edit this (lower = more focused, higher = more random)

print(f"Generating from: '{YOUR_PROMPT}'\n")
print("=" * 60)
result = generate_text(YOUR_PROMPT, max_tokens=MAX_TOKENS, top_k=TOP_K)
print(result)
print("=" * 60)

## Step 8: Experiment with Model Configuration

Try training different model sizes and architectures.

In [ ]:
# Create custom configurations
print("Available BDHConfig parameters:")
print("=" * 60)

default_config = bdh.BDHConfig()
for key, value in vars(default_config).items():
    print(f"{key:20s}: {value}")

print("\n" + "=" * 60)
print("Experiment ideas:")
print("=" * 60)
print("1. Increase n_embd for larger model")
print("2. Increase n_layer for deeper model")
print("3. Adjust dropout for regularization")
print("4. Change vocab_size for different tokenization")
print("\nExample:")
print("  custom_config = bdh.BDHConfig(n_embd=512, n_layer=8)")
print("  custom_model = bdh.BDH(custom_config).to(device)")

## Step 9: Save Your Model

In [ ]:
# Save model checkpoint
checkpoint_path = "bdh_model.pt"

torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'config': config,
    'final_loss': losses[-1] if losses else None,
}, checkpoint_path)

print(f"Model saved to {checkpoint_path}")
print("\nTo download: Click the folder icon on the left, right-click the file, and select 'Download'")

## Step 10: Load a Saved Model

In [ ]:
# Load model from checkpoint
def load_model(checkpoint_path):
    checkpoint = torch.load(checkpoint_path)
    
    loaded_config = checkpoint['config']
    loaded_model = bdh.BDH(loaded_config).to(device)
    loaded_model.load_state_dict(checkpoint['model_state_dict'])
    loaded_model.eval()
    
    print(f"Model loaded from {checkpoint_path}")
    print(f"Final training loss: {checkpoint.get('final_loss', 'N/A')}")
    
    return loaded_model

# Example usage:
# loaded_model = load_model('bdh_model.pt')

## Next Steps

### Learn More:
1. Read the paper: [arXiv:2509.26507](https://arxiv.org/abs/2509.26507)
2. Watch: [SuperDataScience podcast](https://www.youtube.com/watch?v=mfV44-mtg7c)
3. Explore the code: `bdh.py` contains the full model implementation

### Experiments to Try:
1. **Different datasets**: Train on different text (books, code, etc.)
2. **Hyperparameter tuning**: Experiment with learning rate, batch size
3. **Architecture changes**: Modify `BDHConfig` parameters
4. **Longer training**: Increase `MAX_ITERS` for better performance
5. **Temperature sampling**: Adjust generation randomness

### Advanced:
- Implement custom attention visualization
- Add evaluation metrics (perplexity, BLEU)
- Fine-tune on domain-specific text
- Compare with standard Transformer

**Happy experimenting! 🐉**